<a href="https://colab.research.google.com/github/kalaiarasi818/earnings-tone-dashboard/blob/main/notebooks/day1_tone_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Hello, tone project!")

Hello, tone project!


In [2]:
sentences = [
    "We delivered record revenue this quarter and expect strong growth to continue.",
    "Margins came under pressure due to rising input costs.",
    "We cannot say with certainty when demand will recover.",
    "Our order book is healthy and customer response has been excellent.",
    "We are not concerned about the competitive environment.",
    "The board approved a dividend of ₹5 per share.",
    "We hope to see some improvement in the second half, although visibility is limited.",
    "Net profit fell 18 percent compared with the same quarter last year.",
    "We believe costs may stabilize, but it is difficult to predict.",
    "We are confident in our guidance and remain on track to meet it.",
]

my_labels = ["C", "W", "W", "C", "C", "C", "N", "W", "N", "C"]

print(len(sentences), len(my_labels))

10 10


In [3]:
positive  = ["record", "strong", "growth", "healthy", "excellent",
             "confident", "improvement", "delivered", "stabilize"]
negative  = ["pressure", "fell", "decline", "weak", "loss",
             "concerned", "drop", "slowdown"]
uncertain = ["may", "might", "hope", "believe", "cannot", "limited",
             "difficult", "predict", "uncertain", "approximately"]

In [4]:
import re

def count_words(text):
    words = re.findall(r"[a-z']+", text.lower())
    pos = sum(w in positive for w in words)
    neg = sum(w in negative for w in words)
    unc = sum(w in uncertain for w in words)
    return pos, neg, unc

def to_label(pos, neg, unc):
    score = pos - neg - unc
    if score >= 1:
        return "C"
    if score <= -1:
        return "W"
    return "N"

matches = 0
for i, (s, mine) in enumerate(zip(sentences, my_labels), start=1):
    p, n, u = count_words(s)
    guess = to_label(p, n, u)
    ok = guess == mine
    matches += ok
    print(f"{i:>2} | pos={p} neg={n} unc={u} | wordlist={guess} | me={mine} | {'OK' if ok else 'DIFFERENT'}")

print(f"\nAgreement with me: {matches}/10")

 1 | pos=4 neg=0 unc=0 | wordlist=C | me=C | OK
 2 | pos=0 neg=1 unc=0 | wordlist=W | me=W | OK
 3 | pos=0 neg=0 unc=1 | wordlist=W | me=W | OK
 4 | pos=2 neg=0 unc=0 | wordlist=C | me=C | OK
 5 | pos=0 neg=1 unc=0 | wordlist=W | me=C | DIFFERENT
 6 | pos=0 neg=0 unc=0 | wordlist=N | me=C | DIFFERENT
 7 | pos=1 neg=0 unc=2 | wordlist=W | me=N | DIFFERENT
 8 | pos=0 neg=1 unc=0 | wordlist=W | me=W | OK
 9 | pos=1 neg=0 unc=4 | wordlist=W | me=N | DIFFERENT
10 | pos=1 neg=0 unc=0 | wordlist=C | me=C | OK

Agreement with me: 6/10


In [5]:
from transformers import pipeline

finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)
results = finbert(sentences)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [6]:
label_map = {"positive": "C", "negative": "W", "neutral": "N"}

matches = 0
for i, (s, mine, scores) in enumerate(zip(sentences, my_labels, results), start=1):
    best = max(scores, key=lambda d: d["score"])
    guess = label_map[best["label"]]
    ok = guess == mine
    matches += ok
    print(f"{i:>2} | finbert={best['label']:<8} ({best['score']:.2f}) | label={guess} | me={mine} | {'OK' if ok else 'DIFFERENT'}")

print(f"\nAgreement with me: {matches}/10")

 1 | finbert=positive (0.95) | label=C | me=C | OK
 2 | finbert=negative (0.97) | label=W | me=W | OK
 3 | finbert=neutral  (0.70) | label=N | me=W | DIFFERENT
 4 | finbert=positive (0.95) | label=C | me=C | OK
 5 | finbert=neutral  (0.89) | label=N | me=C | DIFFERENT
 6 | finbert=positive (0.64) | label=C | me=C | OK
 7 | finbert=positive (0.88) | label=C | me=N | DIFFERENT
 8 | finbert=negative (0.98) | label=W | me=W | OK
 9 | finbert=positive (0.82) | label=C | me=N | DIFFERENT
10 | finbert=positive (0.91) | label=C | me=C | OK

Agreement with me: 6/10


## Day 1 notes

- Today I learned what an earnings call is and what "tone" means.
- I compared my own labels with two methods: a word list and FinBERT. Both agreed with me on 6 out of 10 sentences, but they missed different sentences.
- I don't trust either method yet, and I don't fully trust my own labels either. This is my first time comparing my answers with AI answers, so for me today was about experience.
- I can't say which method is better yet. I'll decide once I have more experience and more data to test on.
- Next, I want to try this on real transcripts.